In [2]:
import sys
# Add parent directory to path to import file_paths module
sys.path.insert(0, '..')
import file_paths
import helper_functions

import eelbrain

SUBJECTS = helper_functions.get_subjects()

In [3]:
# Loop through subjects to estimate TRFs

for SUBJECT in SUBJECTS:
    print("-" * 50)
    print(f"Processing subject {SUBJECT}...")
    
    # Check if all TRF files already exist
    trf_dir = file_paths.TRF_DIR / f"{SUBJECT}"
    trf_attended_path = trf_dir / f"{SUBJECT}_attended_envelope_onset_trf.pickle"
    trf_unattended_path = trf_dir / f"{SUBJECT}_unattended_envelope_onset_trf.pickle"
    
    if all([
        trf_attended_path.exists(), 
        trf_unattended_path.exists(), 
    ]):
        print(f"All TRF files for {SUBJECT} already exist, skipping.")
        continue
    
    # Load EEG data from pickle files
    eeg_path = file_paths.EEG_DIR / SUBJECT / f"{SUBJECT}_eeg.pickle"
    eeg = eelbrain.load.unpickle(eeg_path)


    # Load envelope data (all 4 versions)
    attended_envelope_path = file_paths.ENVELOPES_DIR / SUBJECT / f"{SUBJECT}_attended_envelope.pickle"
    unattended_envelope_path = file_paths.ENVELOPES_DIR / SUBJECT / f"{SUBJECT}_unattended_envelope.pickle"
    attended_envelope_onset_path = file_paths.ENVELOPES_DIR / SUBJECT / f"{SUBJECT}_attended_envelope_onset.pickle"
    unattended_envelope_onset_path = file_paths.ENVELOPES_DIR / SUBJECT / f"{SUBJECT}_unattended_envelope_onset.pickle"
    
    attended_envelope = eelbrain.load.unpickle(attended_envelope_path)
    unattended_envelope = eelbrain.load.unpickle(unattended_envelope_path)
    attended_envelope_onset = eelbrain.load.unpickle(attended_envelope_onset_path)
    unattended_envelope_onset = eelbrain.load.unpickle(unattended_envelope_onset_path)


    print(f"EEG: {eeg}")
    print(f"Attended onset envelope: {attended_envelope}")
    print(f"Unattended onset envelope: {unattended_envelope}")

    # Estimate TRFs for attended and unattended conditions using boosting
    print("Estimating attended onset TRF...")
    trf_attended = eelbrain.boosting(
            eeg,
            [attended_envelope, attended_envelope_onset],
            -0.100,
            1.000,
            error='l1',
            basis=0.050,
            partitions=5,
            test=1, # use cross-validation
            selective_stopping=True
        )
    
    print("Estimating unattended onset TRF...")
    trf_unattended = eelbrain.boosting(
            eeg,
            [unattended_envelope, unattended_envelope_onset],
            -0.100,
            1.000,
            error='l1',
            basis=0.050,
            partitions=5,
            test=1,
            selective_stopping=True
        )
    

    # Save all 4 TRFs
    trf_dir.mkdir(exist_ok=True, parents=True)
    
    eelbrain.save.pickle(trf_attended, trf_attended_path)
    eelbrain.save.pickle(trf_unattended, trf_unattended_path)

    print(f"Saved attended envelope + onset TRF to {trf_attended_path}")
    print(f"Saved unattended envelope + onset TRF to {trf_unattended_path}")

--------------------------------------------------
Processing subject S1...
EEG: <NDVar: 64 sensor, 192000 time>
Attended onset envelope: <NDVar 'attended': 192000 time>
Unattended onset envelope: <NDVar 'unattended': 192000 time>
Estimating attended onset TRF...
Estimating unattended onset TRF...
Saved attended envelope + onset TRF to /Users/sylvestereley/Data/cocoha3/TRFs/S1/S1_attended_envelope_onset_trf.pickle
Saved unattended envelope + onset TRF to /Users/sylvestereley/Data/cocoha3/TRFs/S1/S1_unattended_envelope_onset_trf.pickle
--------------------------------------------------
Processing subject S2...
EEG: <NDVar: 64 sensor, 192000 time>
Attended onset envelope: <NDVar 'attended': 192000 time>
Unattended onset envelope: <NDVar 'unattended': 192000 time>
Estimating attended onset TRF...
Estimating unattended onset TRF...
Saved attended envelope + onset TRF to /Users/sylvestereley/Data/cocoha3/TRFs/S2/S2_attended_envelope_onset_trf.pickle
Saved unattended envelope + onset TRF to 